# Session 20: Neural networks and deep learning, without the maths wall

Build a neuron and a training loop with NumPy, record four TensorFlow Playground challenges, then fit a network on the loans and compare it with the forest. Your scores will land inside the ranges in the checks and differ a little from the class numbers.

**SOLUTION VERSION. Tutor only.** Run the cells from the top. Each cell that starts with '# Your turn' is yours to complete. Run it, then run the check cell after it: the check tells you if you are right and what to fix.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, zipfile, urllib.request
DATA_ZIP_URL = ""   # your tutor may give you a link; leave empty to upload data.zip instead
if not os.path.exists("orders.csv"):
    if DATA_ZIP_URL:
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

## Task 1. One neuron
Write a function `neuron(x, w, b)` that returns the sigmoid of the weighted sum plus the bias. Test it on a loan with inputs `[1, 0.8, 0.3]`, weights `[0.9, -1.2, 1.5]` and bias `-1.0`.

In [ ]:
import numpy as np

def neuron(x, w, b):
    z = np.dot(x, w) + b
    return 1 / (1 + np.exp(-z))

score = neuron(np.array([1, 0.8, 0.3]), np.array([0.9, -1.2, 1.5]), -1.0)
print(round(score, 3))

In [ ]:
assert score is not None, 'neuron returns None. Replace the return None line with the sigmoid of the weighted sum plus b'
assert abs(score - 0.352) < 0.001, 'The score should be about 0.352. Check np.dot, the bias and the sigmoid formula 1 / (1 + exp(-z))'
print('Task 1 correct')

## Task 2. Change the weights
The loan in Task 1 scores about 0.35. Change the first weight so the same loan scores above 0.9. Store the new score in `new_score`.

In [ ]:
new_score = neuron(np.array([1, 0.8, 0.3]), np.array([4.5, -1.2, 1.5]), -1.0)
print(round(new_score, 3))

In [ ]:
assert new_score > 0.9, 'new_score should be above 0.9. Call neuron again with a bigger first weight and keep the other numbers'
print('Task 2 correct')

## Task 3. A training loop
One weight must learn that 2 times the weight gives 6. Start at `w = 0.0`, use a learning rate of 0.05 and run 8 steps. Store the final weight in `w`.

In [ ]:
x, target = 2.0, 6.0
w, lr = 0.0, 0.05
for step in range(8):
    pred = w * x
    grad = 2 * (pred - target) * x
    w = w - lr * grad
print(round(w, 3))

In [ ]:
assert 2.9 < w < 3.0, 'After 8 steps w should be close to 3 but still below it. Move w against the slope: w = w - lr * slope'
print('Task 3 correct')

## Task 4. A learning rate that is too big
Run the same loop with `lr = 0.5` and store the final loss in `loss_big`, where the loss is `(w * x - target) ** 2`. Print the weight at each step, then add a text cell and write down what happens.

In [ ]:
w, lr = 0.0, 0.5
for step in range(8):
    pred = w * x
    grad = 2 * (pred - target) * x
    w = w - lr * grad
loss_big = (w * x - target) ** 2
print(round(loss_big, 1))

In [ ]:
assert loss_big > 100, 'loss_big should be above 100. With lr = 0.5 each step overshoots, so run all 8 steps and use the final w'
print('Task 4 correct')

## Task 5. Playground results
Open TensorFlow Playground in your browser. Add a text cell below and record each challenge in your own words: what you changed and what you saw.

1. Circle, one hidden layer: 
2. Spiral, small network then bigger: 
3. Noise raised, overfitting: 
4. Learning rate and activation changed: 

## Task 6. Prepare the loan data
Load `loans.csv`. Keep ages from 18 to 100. Drop the column `collection_calls`, which leaks the answer. Build `X` from the remaining numeric columns and `y` from `default`, then split 75 and 25 with `random_state=42`.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

loans = pd.read_csv('loans.csv')
loans = loans[(loans.age >= 18) & (loans.age <= 100)]
cols = ['age', 'monthly_income_ugx', 'loan_amount_ugx', 'term_months',
        'previous_loans', 'previous_late_payments', 'group_loan',
        'savings_balance_ugx', 'has_mobile_money', 'distance_to_branch_km']
X = loans[cols]
y = loans['default']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)
print(X_train.shape, X_test.shape)

In [ ]:
assert 'collection_calls' not in X.columns, 'Leave collection_calls out of X: it leaks the answer'
assert 7900 < len(X) < 8000, 'Keep ages 18 to 100 and put the ten numeric columns in cols (X should have just under 8,000 rows)'
assert abs(len(X_test) / len(X) - 0.25) < 0.01, 'Split 75 and 25 with test_size=0.25'
print('Task 6 correct')

## Task 7. Fit the network
Build a pipeline of a median imputer, a standard scaler and `MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=500, random_state=42)`. Fit it, then store the AUC on the test rows in `auc_mlp`.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score

mlp = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=500, random_state=42))
mlp.fit(X_train, y_train)
auc_mlp = roc_auc_score(y_test, mlp.predict_proba(X_test)[:, 1])
print(round(auc_mlp, 3))

In [ ]:
assert 0.65 <= auc_mlp <= 0.85, 'auc_mlp should be between 0.65 and 0.85. Did you scale, fit on the training rows, and score the test rows with predict_proba?'
print('Task 7 correct')

## Task 8. Compare with the forest
Fit the random forest from Session 17 on the same rows, using the same imputer, and store its test AUC in `auc_rf`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = make_pipeline(
    SimpleImputer(strategy='median'),
    RandomForestClassifier(n_estimators=200, random_state=42))
rf.fit(X_train, y_train)
auc_rf = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
print(round(auc_mlp, 3), round(auc_rf, 3))

In [ ]:
assert 0.65 <= auc_rf <= 0.85, 'auc_rf should be between 0.65 and 0.85. Score the test rows with predict_proba'
assert abs(auc_mlp - auc_rf) < 0.12, 'The two AUCs should be within 0.12 of each other. Check both used the same split'
print('Task 8 correct')

## Task 9. Reflect
Write one sentence: which model would you give a loans manager, and why? Mention the scores, the effort to tune and how easily each can be explained.

## Task 10 (optional). A pretrained model, used and checked
This cell needs Colab and an internet connection, and it opens a second file picker for your photo. Upload one photo of an everyday object. Run the pipeline from the slide, read the three labels, and write one sentence on whether you would trust them.

In [ ]:
from google.colab import files
from transformers import pipeline

name = list(files.upload())[0]
classifier = pipeline('image-classification')
print(classifier(name)[:3])